In [ ]:
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

In [1]:
from snowflake.snowpark import Session
session = Session.builder.configs({
    "account": "CQOUAQZ-REVEEL_AZURE", 
    "user":"RAMYA-SQUADRON@REVEELGROUP.COM", # e.g. xy12345.ca-central-1
    "warehouse": "DEV_WH",
    "database": "STAGING",
    "schema": "AUDIT",
    "role": "SYSADMIN",
    "authenticator": "externalbrowser",
    "client_session_keep_alive": True
}).create()


import sys
sys.path.append("C:/Users/nramy/OneDrive/Documents/Reveel/Snowflake")


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/fe7e6f0e-be6c-4944-b970-ff6d27342141/saml2?SAMLRequest=nZJLc9owFEb%2Fikdd25aNeVgDZFweU6cUCOBMyqYjbBnUyJIryRj66ysMzKSLZJGdRjr36kjf7T%2BcCmYdiVRU8AHwHAgswlORUb4fgGQztXvAUhrzDDPByQCciQIPw77CBStRVOkDX5E%2FFVHaMo24Qs3BAFSSI4EVVYjjgiikU7SOfsyQ70BUSqFFKhh4U%2FJxBVaKSG0M7yWZokbvoHWJXLeua6duOULuXR9C6MLQNdQF%2BXLnT%2BZN7%2FCeC4MLbwiDL29uXym%2FfsFHWrsrpNC3zWZpLxfrDbCiu%2BpIcFUVRK6JPNKUJKvZVUAZg9HTIometvZq8jyZzH5F22Q1cRQXdc7wK0lFUVbaNHbMys1J5jKxp%2Bbt8XgAylea7cLvjzw%2B7vPT%2FPR4PtbLth%2Bc%2BD75TWN6aE0Xfxelmr9o2PuZpMB6vofrX8KNlapIzC%2BRarMF%2FbbtQdvvbrwQwR7yQ6fd6WyBNTaRUo51U3n3bjycgqZSKJFrwRnl5GbZJZ0cEntHOqkdhEFg78IutPO8k%2FndVuB7gedegvbBdXhQIyKHn%2FiSvvu2wW0U5yadeLwUjKZnaypkgfX74XmO1%2BzQzM4bFJECUxZlmSRKmRAZE%2FVIEqzNxG

In [2]:
# Snowflake notebook source
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
from snowflake.snowpark.exceptions import SnowparkSQLException

In [3]:
# from notebooks.misc_utils import (
#     get_all_day_paths, 
#     get_upload_cols, 
#     filter_new_files, 
#     create_dataframe_from_schema, 
#     read_files_with_filename
# )
from notebooks import sfutils
from notebooks.utils.edi_schemas import FEDEX_RAW_EDI_JSON_SCHEMA
from notebooks.utils.table_names import (
    FEDEX_EDI_110_4010_RAW,
    FEDEX_EDI_110_4060_RAW,
    FEDEX_EDI_210_RAW,
)
from notebooks.etl.widgets import catalog
#from notebooks.etl.invoice.universal_loader import azure_url

from notebooks.misc_utils import read_files_with_filename

In [4]:
# sfutils.widgets.text("days_before", "1") ---revert back - from code 
# sfutils.widgets.text("overwrite_schema", "false") --revert back from code 

sfutils.widgets.text("days_before", "-1") 
sfutils.widgets.text("overwrite_schema", "true") 

'true'

In [5]:
days_before = sfutils.widgets.get("days_before")
overwrite_schema = sfutils.widgets.get("overwrite_schema")

In [6]:
#days_before = -1
days_before = int(days_before)
print(f"days_before: {days_before}")

days_before: -1


In [7]:
print(days_before, overwrite_schema, catalog)

-1 true staging


In [ ]:
# sfutils.task_values.set(key="days_before", value=days_before)
# sfutils.task_values.set(key="overwrite_schema", value=overwrite_schema)



In [ ]:
# fedex_load_type = sfutils.jobs.taskValues.get(
#     taskKey="fedex_edi_load", key="days_before", debugValue=days_before
# )

In [ ]:
# overwrite_schema = sfutils.jobs.taskValues.get(
#     taskKey="fedex_edi_load", key="overwrite_schema", debugValue="false"
# )

In [ ]:
# overwrite_mode = "overwrite" if days_before <= 0 else "append"
# print(overwrite_mode) ###NOt needed 

In [7]:
azure_url = {
    f"@{catalog}.public.sf_stage_invoice" : "azure://stpreproccarrinvprdwu01.blob.core.windows.net/maincontainer/",
    f"@{catalog}.public.sf_stage_edi" : "azure://stinbdataediprdwu01.blob.core.windows.net/maincontainer/",
    f"@{catalog}.public.sf_stage" : "azure://stcarrshiptrckapiprdwu01.blob.core.windows.net/maincontainer/"
}

In [9]:

sfutils.widgets.text("stage_name", "sf_stage_edi")
stage_name = sfutils.widgets.get("stage_name")

stage_suffix = f"@{catalog}.public.{stage_name}"
azure_url_path = azure_url[stage_suffix]

print(overwrite_schema, stage_suffix, azure_url_path)


true @staging.public.sf_stage_edi azure://stinbdataediprdwu01.blob.core.windows.net/maincontainer/


In [10]:

stage = "STAGING.PUBLIC.SF_STAGE_EDI"
carrier = "fedex"
entire_path = f"{stage}/carrier={carrier}/"
location = f"carrier={carrier}"

In [11]:
from snowflake.snowpark import DataFrame, Session
from snowflake.snowpark.types import *
from snowflake.snowpark.functions import col

def create_dataframe_from_schema(
    session: Session,
    schema: StructType,
    stage_name: str,
    file_format: str = 'ndjson_format',
    clean_column_names: bool = True,
    azure_url_path: str = 'None',
) -> DataFrame:
    """
    Creates DataFrame with full schema enforcement on transactionSets.
    Uses OBJECT_CONSTRUCT_KEEP_NULL to ensure NULL fields are included.
    """
    
    # Extract schemas from transactionSets
    detail_schema = None
    heading_schema = None
    summary_schema = None
    
    for field in schema.fields:
        if field.name.strip('"').strip("'") == "transactionSets":
            if isinstance(field.datatype, ArrayType):
                element_type = field.datatype.element_type
                if isinstance(element_type, StructType):
                    for nested_field in element_type.fields:
                        nested_key = nested_field.name.strip('"').strip("'")
                        if nested_key == "detail":
                            detail_schema = nested_field.datatype
                        elif nested_key == "heading":
                            heading_schema = nested_field.datatype
                        elif nested_key == "summary":
                            summary_schema = nested_field.datatype
    
    def build_object_construct_for_struct(base_path: str, struct_schema: StructType, depth: int = 0) -> str:
        """
        Builds OBJECT_CONSTRUCT_KEEP_NULL for a StructType with all fields.
        CRITICAL: Using KEEP_NULL to ensure NULL fields are included!
        """
        if depth > 5:  # Safety limit
            return f"PARSE_JSON($1):{base_path}"
        
        field_parts = []
        
        for field in struct_schema.fields:
            field_key = field.name.strip('"').strip("'")
            field_path = f"{base_path}:{field_key}"
            field_type = field.datatype
            
            if isinstance(field_type, StringType):
                value_expr = f"PARSE_JSON($1):{field_path}::STRING"
            elif isinstance(field_type, (IntegerType, LongType)):
                value_expr = f"PARSE_JSON($1):{field_path}::NUMBER(38,0)"
            elif isinstance(field_type, DoubleType):
                value_expr = f"PARSE_JSON($1):{field_path}::FLOAT"
            elif isinstance(field_type, BooleanType):
                value_expr = f"PARSE_JSON($1):{field_path}::BOOLEAN"
            elif isinstance(field_type, TimestampType):
                value_expr = f"PARSE_JSON($1):{field_path}::TIMESTAMP"
            elif isinstance(field_type, DateType):
                value_expr = f"PARSE_JSON($1):{field_path}::DATE"
            elif isinstance(field_type, StructType):
                # Recursively build nested struct
                value_expr = build_object_construct_for_struct(field_path, field_type, depth + 1)
            elif isinstance(field_type, ArrayType):
                # Keep arrays as VARIANT
                value_expr = f"PARSE_JSON($1):{field_path}"
            else:
                # VariantType or unknown
                value_expr = f"PARSE_JSON($1):{field_path}"
            
            field_parts.append(f"'{field_key}', {value_expr}")
        
        # CRITICAL: Use OBJECT_CONSTRUCT_KEEP_NULL
        return f"OBJECT_CONSTRUCT_KEEP_NULL({', '.join(field_parts)})"
    
    # Build field selections (simple approach for top-level fields)
    field_selections = []
    
    for field in schema.fields:
        field_name = field.name
        json_key = field.name.strip('"').strip("'")
        field_type = field.datatype
        
        print(f"Processing field: {field_name}")
        
        # Simple types
        if isinstance(field_type, StringType):
            sql_type = f"PARSE_JSON($1):{json_key}::STRING"
        elif isinstance(field_type, (IntegerType, LongType)):
            sql_type = f"PARSE_JSON($1):{json_key}::NUMBER(38,0)"
        elif isinstance(field_type, DoubleType):
            sql_type = f"PARSE_JSON($1):{json_key}::FLOAT"
        elif isinstance(field_type, BooleanType):
            sql_type = f"PARSE_JSON($1):{json_key}::BOOLEAN"
        elif isinstance(field_type, TimestampType):
            sql_type = f"PARSE_JSON($1):{json_key}::TIMESTAMP"
        elif isinstance(field_type, DateType):
            sql_type = f"PARSE_JSON($1):{json_key}::DATE"
        else:
            # Complex types - keep as VARIANT for now
            sql_type = f"PARSE_JSON($1):{json_key}"
        
        field_selections.append(f"{sql_type} as {field_name}")
    
    # Build schema-enforced transactionSets
    if detail_schema and heading_schema and summary_schema:
        print("="*80)
        print("Building schema-enforced transactionSets with OBJECT_CONSTRUCT_KEEP_NULL...")
        print("="*80)
        
        # Build OBJECT_CONSTRUCT_KEEP_NULL for each section
        print("Building detail construct...")
        detail_construct = build_object_construct_for_struct("transactionSets[0]:detail", detail_schema)
        
        print("Building heading construct...")
        heading_construct = build_object_construct_for_struct("transactionSets[0]:heading", heading_schema)
        
        print("Building summary construct...")
        summary_construct = build_object_construct_for_struct("transactionSets[0]:summary", summary_schema)
        
        # Build final SQL with OBJECT_CONSTRUCT_KEEP_NULL
        sql_query = f"""
            SELECT 
                PARSE_JSON($1):delimiters as "delimiters",
                PARSE_JSON($1):envelope as "envelope",
                PARSE_JSON(
                    ARRAY_CONSTRUCT(
                        OBJECT_CONSTRUCT_KEEP_NULL(
                            'detail', {detail_construct},
                            'heading', {heading_construct},
                            'summary', {summary_construct}
                        )
                    )::STRING
                ) as "transactionSets",
                PARSE_JSON($1):upload_year::NUMBER(38,0) as "upload_year",
                PARSE_JSON($1):upload_month::NUMBER(38,0) as "upload_month",
                PARSE_JSON($1):upload_day::NUMBER(38,0) as "upload_day",
                OBJECT_CONSTRUCT('file_path', metadata$filename, 'file_name', SPLIT_PART(metadata$filename, '/', -1), 'file_row_number', metadata$file_row_number, 'file_content_key', metadata$file_content_key, 'file_modification_time', metadata$file_last_modified) as "_metadata",
                CONCAT('{azure_url_path}', METADATA$FILENAME) as filename
            FROM @{stage_name}
            (FILE_FORMAT => '{file_format}')
        """
    else:
        # Fallback
        sql_query = f"""
            SELECT 
                {schema_fields},
                OBJECT_CONSTRUCT('file_path', metadata$filename, 'file_name', SPLIT_PART(metadata$filename, '/', -1), 'file_row_number', metadata$file_row_number, 'file_content_key', metadata$file_content_key, 'file_modification_time', metadata$file_last_modified) as "_metadata",
                CONCAT('{azure_url_path}', METADATA$FILENAME) as filename
            FROM @{stage_name}
            (FILE_FORMAT => '{file_format}')
        """
    
    print("="*80)
    print(f"GENERATED SQL LENGTH: {len(sql_query):,} characters")
    print("="*80)
    print("SQL Preview (first 3000 chars):")
    print(sql_query[:3000])
    print("...")
    print("="*80)
    
    print(f"Executing query against stage: {stage_name}")
    df = session.sql(sql_query)
    
    if clean_column_names:
        print("Cleaning column names...")
        df.create_or_replace_temp_view("temp_df")
        
        select_parts = []
        for col_name in df.columns:
            clean_name = col_name.replace('"', '').replace("'", "")
            select_parts.append(f'"{col_name}" as {clean_name}')
        
        select_sql = ", ".join(select_parts)
        df = session.sql(f"SELECT {select_sql} FROM temp_df")
    
    print(f"✅ Created DataFrame with {len(df.columns)} columns")
    print("✅ NULL fields will be KEPT in the VARIANT (not omitted)")
    return df

In [12]:
try:
    if days_before > 0:
        df = read_files_with_filename(
                session=session,
                stage_name=stage,
                prefix=location,
                days_before=days_before,
                include_current_day=True,
                file_format="test_ndjson_format",
                clean_column_names=True,
                schema=FEDEX_RAW_EDI_JSON_SCHEMA,
        )
        print("inc")
        df.show(2)
    else:
        print("history")
        df = create_dataframe_from_schema(
            session=session,
            schema=FEDEX_RAW_EDI_JSON_SCHEMA, 
            stage_name=entire_path, 
            file_format="test_ndjson_format", 
            clean_column_names=False,
            azure_url_path=azure_url_path
        )
except SnowparkSQLException as e:
    if e.error_code == 1304:
        print(f"No files found (error 1304), creating empty DataFrame")
        df = session.create_dataframe([], FEDEX_RAW_EDI_JSON_SCHEMA)

    else:
        print(f"Other SnowparkSQLException: {e.error_code} - {e}")
        raise

history
Processing field: "delimiters"
Processing field: "envelope"
Processing field: "transactionSets"
Processing field: "upload_year"
Processing field: "upload_month"
Processing field: "upload_day"
Building schema-enforced transactionSets with OBJECT_CONSTRUCT_KEEP_NULL...
Building detail construct...
Building heading construct...
Building summary construct...
GENERATED SQL LENGTH: 5,562 characters
SQL Preview (first 3000 chars):

            SELECT 
                PARSE_JSON($1):delimiters as "delimiters",
                PARSE_JSON($1):envelope as "envelope",
                PARSE_JSON(
                    ARRAY_CONSTRUCT(
                        OBJECT_CONSTRUCT_KEEP_NULL(
                            'detail', OBJECT_CONSTRUCT_KEEP_NULL('assigned_number_LX_loop', PARSE_JSON($1):transactionSets[0]:detail:assigned_number_LX_loop, 'transaction_set_line_number_0400_loop', PARSE_JSON($1):transactionSets[0]:detail:transaction_set_line_number_0400_loop, 'transaction_set_line_number_LX_l

In [13]:
df.print_schema()

root
 |-- "delimiters": VariantType() (nullable = True)
 |-- "envelope": VariantType() (nullable = True)
 |-- "transactionSets": VariantType() (nullable = True)
 |-- "upload_year": LongType() (nullable = True)
 |-- "upload_month": LongType() (nullable = True)
 |-- "upload_day": LongType() (nullable = True)
 |-- "_metadata": MapType (nullable = True)
 |   |-- key: StringType()
 |   |-- value: StringType()
 |-- "FILENAME": StringType(16777280) (nullable = True)


In [14]:
df.show(2,max_width=100000)

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"delimiters"          |"envelope"                                      |"transactionSets"                                                                  |"upload_year"  |"upload_month"  |"upload_day"  |"_metadata"                                                                                                              |"FILENAME"                                                                                                                                                             |
--------

In [ ]:
#df.print_schema()
df=df.filter(F.col('"upload_year"') == 2024).limit(100)
#df.select('"transactionSets"').show(10,max_width=100000)



In [ ]:
df.where('"filename"=')

In [ ]:
df = get_upload_cols(df)
#df.show(1)

In [ ]:
df.printSchema()

In [ ]:
df=df.filter(F.col("upload_year") == 2024)

In [ ]:
df = df.withColumn(
    "edi_type",
    F.col('"transactionSets"')[0]["heading"]["transaction_set_header_ST"]["transaction_set_identifier_code_01"].cast(StringType())
)
df = df.withColumn('"_metadata"', F.col('"_metadata"').cast(VariantType()))

#df.select('"edi_type"').show(n=1, max_width=100000)
#df.printSchema()


In [ ]:
raw_110_4060 = df.where(
    (F.col("edi_type") == "110")
    & (F.col('"envelope"')["interchangeHeader"]["controlVersionNumber"] == "00406")
)
raw_110_4010 = df.where(
    (F.col("edi_type") == "110")
    & (F.col('"envelope"')["interchangeHeader"]["controlVersionNumber"] == "00401")
)
raw_210 = df.where(F.col("edi_type") == "210")

In [ ]:
raw_210.printSchema()

In [ ]:
raw_210.show(2)

In [ ]:
raw_110_4010.show(1)


In [ ]:
raw_110_4060.show(1)

In [ ]:
if days_before > 0:
    raw_110_4060 = filter_new_files(raw_110_4060, session.table(FEDEX_EDI_110_4060_RAW))
    raw_110_4010 = filter_new_files(raw_110_4010, session.table(FEDEX_EDI_110_4010_RAW))
    raw_210 = filter_new_files(raw_210, session.table(FEDEX_EDI_210_RAW))

print(f"raw_110_4060 count: {raw_110_4060.count()}")
print(f"raw_110_4010 count: {raw_110_4010.count()}")
print(f"raw_210 count: {raw_210.count()}")

In [ ]:
# raw_110_4060.write.mode(overwrite_mode).option(
#     "overwriteSchema", overwrite_schema
# ).partitionBy('"upload_year"', '"upload_month"').saveAsTable(FEDEX_EDI_110_4060_RAW)
if overwrite_schema:
    print("true")
    # Recreate table from the DF schema + data (schema overwrite)
    raw_110_4060.write.mode("overwrite").save_as_table(FEDEX_EDI_110_4060_RAW)
else:
    # Truncate + insert (keeps table properties/privs intact)
    session.sql(f"TRUNCATE TABLE {FEDEX_EDI_110_4060_RAW}").collect()
    raw_110_4060.write.mode("append").save_as_table(FEDEX_EDI_110_4060_RAW)

In [ ]:
# raw_110_4010.write.mode(overwrite_mode).option(
#     "overwriteSchema", overwrite_schema
# ).partitionBy('"upload_year"', '"upload_month"').saveAsTable(FEDEX_EDI_110_4010_RAW)
if overwrite_schema:
    print("true")
    # Recreate table from the DF schema + data (schema overwrite)
    raw_110_4010.write.mode("overwrite").save_as_table(FEDEX_EDI_110_4010_RAW)
else:
    # Truncate + insert (keeps table properties/privs intact)
    session.sql(f"TRUNCATE TABLE {FEDEX_EDI_110_4010_RAW}").collect()
    raw_110_4010.write.mode("append").save_as_table(FEDEX_EDI_110_4010_RAW)

In [ ]:
# raw_210.write.mode(overwrite_mode).option(
#     "overwriteSchema", overwrite_schema
# ).partitionBy('"upload_year"', '"upload_month"').saveAsTable(FEDEX_EDI_210_RAW)

if overwrite_schema:
    print("true")
    # Recreate table from the DF schema + data (schema overwrite)
    raw_210.write.mode("overwrite").save_as_table(FEDEX_EDI_210_RAW)
else:
    # Truncate + insert (keeps table properties/privs intact)
    session.sql(f"TRUNCATE TABLE {FEDEX_EDI_210_RAW}").collect()
    raw_210.write.mode("append").save_as_table(FEDEX_EDI_210_RAW)